In [10]:
import pandas as pd
from sqlalchemy import create_engine
import os
import numpy as np

In [ ]:
db_path = os.path.abspath('../database/lending_club.db')
engine = create_engine(f"sqlite:///{db_path}")

query = """
SELECT * FROM raw_loans 
WHERE loan_status IN (
    'Fully Paid', 
    'Charged Off', 
    'Default', 
    'Does not meet the credit policy. Status:Charged Off',
    'Does not meet the credit policy. Status:Fully Paid',
    'Late (31-120 days)'
)
"""

df_bank = pd.read_sql(query, engine)
print(df_bank.shape)
print(df_bank['loan_status'].value_counts())

(237695, 74)
loan_status
Fully Paid                                             184739
Charged Off                                             42475
Late (31-120 days)                                       6900
Does not meet the credit policy. Status:Fully Paid       1988
Default                                                   832
Does not meet the credit policy. Status:Charged Off       761
Name: count, dtype: int64


Из исходной таблицы с ~400000 строк мы отсортировали 237695, то есть только завершенные крединтые истории. Остальные имели стату Current, то есть люди которые сейчас платят кредит, а нам нужны завершенные кейсы.

In [4]:
def create_target(status):
    if status in ['Charged Off', 'Default', 'Late (31-120 days)', 
                  'Does not meet the credit policy. Status:Charged Off']:
        return 1
    else:
        return 0

df_bank['target'] = df_bank['loan_status'].apply(create_target)
print(df_bank['target'].value_counts(normalize=True) * 100)

target
0    78.557395
1    21.442605
Name: proportion, dtype: float64


In [ ]:
df_bank.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237695 entries, 0 to 237694
Data columns (total 75 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           237695 non-null  int64  
 1   member_id                    237695 non-null  int64  
 2   loan_amnt                    237695 non-null  int64  
 3   funded_amnt                  237695 non-null  int64  
 4   funded_amnt_inv              237695 non-null  float64
 5   term                         237695 non-null  object 
 6   int_rate                     237695 non-null  float64
 7   installment                  237695 non-null  float64
 8   grade                        237695 non-null  object 
 9   sub_grade                    237695 non-null  object 
 10  emp_title                    224305 non-null  object 
 11  emp_length                   228539 non-null  object 
 12  home_ownership               237695 non-null  object 
 13 

Next is feature engineering. "emp_length" and "term" should be numeric, and we will define months from "earliest_cr_line".

In [7]:
df_bank['emp_length_int'] = df_bank['emp_length'].str.replace('\+ years', '', regex=True)
df_bank['emp_length_int'] = df_bank['emp_length_int'].str.replace('< 1 year', '0', regex=False)
df_bank['emp_length_int'] = df_bank['emp_length_int'].str.replace(' years', '', regex=False)
df_bank['emp_length_int'] = df_bank['emp_length_int'].str.replace(' year', '', regex=False)
df_bank['emp_length_int'] = pd.to_numeric(df_bank['emp_length_int'], errors='coerce').fillna(0)

In [8]:
df_bank['term_int'] = df_bank['term'].str.replace(' months', '', regex=False).astype(float)
df_bank['annual_inc'] = df_bank['annual_inc'].fillna(df_bank['annual_inc'].median())

In [9]:
df_bank['issue_d']=pd.to_datetime(df_bank['issue_d'], format='%b-%y')
df_bank['earliest_cr_line']=pd.to_datetime(df_bank['earliest_cr_line'], format='%b-%y')

In [12]:
df_bank['months_since_earliest_cr_line'] = round((df_bank['issue_d'] - df_bank['earliest_cr_line']) / np.timedelta64(1, 'D') / 30.44)
max_value = df_bank['months_since_earliest_cr_line'].max()
df_bank.loc[df_bank['months_since_earliest_cr_line'] < 0, 'months_since_earliest_cr_line'] = max_value

In [13]:
to_drop = ['id', 'member_id', 'url', 'desc', 'title', 'next_pymnt_d', 'emp_title', 'zip_code']
df_bank = df_bank.drop(columns=to_drop)